# Phase 7: Hyperparameter Tuning with Optuna

**Goal:** Systematically search for the best XGBoost hyperparameters using Bayesian optimisation (Optuna) with 3-fold stratified cross-validation. Compare tuned vs default model on the held-out test set.

**Why not GridSearch?** Grid search is exhaustive but exponentially expensive. Optuna uses TPE (Tree-structured Parzen Estimator) to sample promising regions of the search space first — typically finds better results in fewer trials.

In [ ]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.data.loader import load_raw
from src.features.engineer import engineer, split
from src.models.train import train_xgboost
from src.models.evaluate import report

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
df = engineer(load_raw())
X_train, X_test, y_train, y_test = split(df)

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {X_train.shape} | scale_pos_weight: {SCALE_POS_WEIGHT:.1f}")

## 1. Default Model Baseline

In [ ]:
default_model = train_xgboost(X_train, y_train)
default_metrics = report(y_test, default_model.predict_proba(X_test)[:, 1])
print("Default params:", default_metrics)

## 2. Optuna Search

Each trial trains XGBoost with a sampled set of hyperparameters and evaluates with 3-fold CV AUC. Optuna learns which regions of the search space are promising and focuses samples there.

In [ ]:
def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 8),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":  SCALE_POS_WEIGHT,
        "eval_metric":       "auc",
        "random_state":      42,
    }
    model = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\nBest CV AUC : {study.best_value:.4f}")
print("Best params :")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## 3. Evaluate Tuned Model on Test Set

In [ ]:
best_params = {**study.best_params,
               "scale_pos_weight": SCALE_POS_WEIGHT,
               "random_state": 42}

tuned_model = xgb.XGBClassifier(**best_params)
tuned_model.fit(X_train, y_train, verbose=False)
tuned_metrics = report(y_test, tuned_model.predict_proba(X_test)[:, 1])

comparison = pd.DataFrame(
    [default_metrics, tuned_metrics],
    index=['Default params', 'Optuna tuned']
)
comparison.style.highlight_max(axis=0, color='#d4edda')

## 4. Optimisation History

In [ ]:
trial_numbers = [t.number for t in study.trials]
trial_values  = [t.value for t in study.trials]
best_so_far   = pd.Series(trial_values).cummax()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(trial_numbers, trial_values, alpha=0.4, s=20, color='#4C72B0', label='Trial AUC')
ax.plot(trial_numbers, best_so_far, color='#DD8452', lw=2, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('CV ROC-AUC')
ax.set_title('Optuna Optimisation History', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Hyperparameter Importances

Which parameters had the most impact on CV AUC? Knowing this helps you focus future tuning efforts.

In [ ]:
importances = optuna.importance.get_param_importances(study)

fig, ax = plt.subplots(figsize=(8, 4))
params = list(importances.keys())
values = list(importances.values())
ax.barh(params[::-1], values[::-1], color='#4C72B0', edgecolor='white')
ax.set_xlabel('Importance')
ax.set_title('Hyperparameter Importances (Optuna FAnova)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Save Tuned Model

In [ ]:
with open('../models/xgboost_tuned.pkl', 'wb') as f:
    pickle.dump(tuned_model, f)

print('Saved → models/xgboost_tuned.pkl')